# Benchmarking Comparison

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 07.04.2026
- Date of Last Modification: 14.04.2026 (by Sebastian Birk (sb75@sanger.ac.uk))

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from pathlib import Path

### 1.2 Define Config

In [ ]:
# ---------------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------------
# Each method is paired with its corresponding embedding key
method_emb_pairs = [
    #("stjepa_v1",      "neighborhood_emb"),
    ("stjepa_v2",      "neighborhood_emb"),
    ("neigh_expr_pca", "X_pca"),
    ("nichecompass",    "nichecompass_latent"),
    ("cellcharter",    "X_cellcharter"),
    ("banksy",         "banksy_latent_harmony"),
    ("graphst",        "emb"),
    ("novae",          "novae_latent_corrected"),
    ("nicheformer",    "X_niche_embeddings"),
    ("CellPLM",        "cellplm_latent"),
    ("scgpt-spatial",  "scgpt_spatial_latent"),
]

### 1.3 Configure Paths

In [ ]:
base_path = Path("/nfs/team361/sb75/stjepa-reproducibility/artifacts/benchmarking")
results_dir = base_path / "results"
figures_dir = base_path / "figures"
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

### 1.4 Run Notebook Setup

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",  # export text as editable text in SVG
})

# Colour-blind friendly palette (Okabe-Ito + extras for 9 methods)
PALETTE = [
    "#73b7a2",
    "#f4a6a6",
    "#d6b36a",
    "#6fb6d8",
    "#b99ad6",
    "#78c7d2",
    "#ffde17",
    "#95a3c3",
    "#ea9776",
    "#db96c0",
]

# Prettier method labels for the plot
METHOD_LABELS = {
    #"stjepa_v1":      "ST-JEPA v1",
    "stjepa_v2":      "TERRA",
    "neigh_expr_pca": "Neigh. Expr. PCA",
    "nichecompass":   "NicheCompass",
    "cellcharter":    "CellCharter",
    "banksy":         "Banksy",
    "graphst":        "GraphST",
    "novae":          "Novae",
    "nicheformer":    "NicheFormer",
    "CellPLM":        "CellPLM",
    "scgpt-spatial":  "scGPT-spatial"
}

# Panel titles and ordering (MERFISH left, STARmap right)
REGION_TITLES = {
    "ccf_region_name":              "Tissue Regions (MERFISH)",
    "Sub_molecular_tissue_region":  "Tissue Regions (STARmap)",
}
REGION_ORDER = ["ccf_region_name", "Sub_molecular_tissue_region"]

### 1.5 Define Functions

In [ ]:
def plot_metric_comparison(
    df: pd.DataFrame,
    metric: str,
    method_order: list[str],
    figsize_per_panel: tuple = (3.0, 2.4),
):
    """
    Bar plot of mean metric per method with individual data points overlaid.
    One panel per region_key. Text exported as editable paths for Illustrator.
    """
    region_keys = REGION_ORDER
    n_panels = len(region_keys)
    fig, axes = plt.subplots(
        1, n_panels,
        figsize=(figsize_per_panel[0] * n_panels, figsize_per_panel[1]),
        sharey=True,
        constrained_layout=True,
    )
    if n_panels == 1:
        axes = [axes]

    n_methods = len(method_order)
    bar_width = 0.6
    colors = PALETTE[:n_methods]

    for ax, rk in zip(axes, region_keys):
        sub = df[df["region_key"] == rk]

        means, sds, all_vals = [], [], []
        for m in method_order:
            vals = sub.loc[sub["method"] == m, metric].values
            means.append(np.mean(vals) if len(vals) > 0 else 0)
            sds.append(np.std(vals) if len(vals) > 0 else 0)
            all_vals.append(vals)

        x = np.arange(n_methods)

        # --- bars (mean) ---
        ax.bar(
            x, means, width=bar_width,
            color=colors, edgecolor="white", linewidth=0.6,
            alpha=0.75, zorder=2,
        )

        # --- error bars (± 1 SD) ---
        ax.errorbar(
            x, means, yerr=sds,
            fmt="none", ecolor="0.25", elinewidth=0.8,
            capsize=2.5, capthick=0.8, zorder=3,
        )

        # --- individual data points (jittered strip) ---
        rng = np.random.default_rng(42)
        for i, vals in enumerate(all_vals):
            n_pts = len(vals)
            if n_pts == 0:
                continue
            jitter = rng.uniform(-0.15, 0.15, size=n_pts)
            ax.scatter(
                np.full(n_pts, i) + jitter, vals,
                s=12, color="0.15", edgecolor="white",
                linewidth=0.3, zorder=4, alpha=0.65,
            )

        # --- dotted reference line at STJEPA v2 level ---
        if "stjepa_v2" in method_order:
            stjepa_idx = method_order.index("stjepa_v2")
            ax.axhline(y=means[stjepa_idx], color="0.3", linestyle="--",
                       linewidth=0.8, zorder=1, alpha=0.7)

        # --- formatting ---
        pretty_rk = REGION_TITLES.get(rk, rk.replace("_", " "))
        ax.set_title(pretty_rk, fontweight="medium")
        ax.set_xticks(x)
        ax.set_xticklabels(
            [METHOD_LABELS.get(m, m) for m in method_order],
            rotation=45, ha="right",
        )
        ax.tick_params(axis="x", pad=1)
        ax.set_ylabel(f"{metric} (\u2191)" if ax is axes[0] else "")
        ax.set_xlim(-0.5, n_methods - 0.5)
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
        ax.grid(axis="y", linewidth=0.3, alpha=0.5)

    return fig

## 2. Analysis

In [ ]:
# ---------------------------------------------------------------------------
# 1. Load and concatenate all CSV files
# ---------------------------------------------------------------------------
frames = []
for method, emb_key in method_emb_pairs:
    fpath = (
        base_path / method
        / f"mmb0-1b_smb1-1b_1p_{emb_key}_niche_identification_metrics_res_sweep.csv"
    )
    if not fpath.exists():
        print(f"[WARN] File not found, skipping: {fpath}")
        continue
    df = pd.read_csv(fpath)
    df["method"] = method
    df["emb_key"] = emb_key
    frames.append(df)

if not frames:
    raise FileNotFoundError("No CSV files were found. Check your paths.")

df_all = pd.concat(frames, ignore_index=True)

# Sanity check: print counts per method and region_key
print("=== Data points per (method, region_key) ===")
print(df_all.groupby(["method", "region_key"]).size().unstack(fill_value=0))
print()

# Save combined dataframe
df_all.to_csv(results_dir / "niche_identification_all_methods.csv", index=False)
print(f"Saved combined dataframe to {results_dir / 'niche_identification_all_methods.csv'}\n")

# ---------------------------------------------------------------------------
# 2. Generate and save figures
# ---------------------------------------------------------------------------
method_order = [m for m, _ in method_emb_pairs]

for metric in ["NMI", "ARI"]:
    fig = plot_metric_comparison(df_all, metric=metric, method_order=method_order)
    for ext in ["svg", "pdf", "png"]:
        out = figures_dir / f"niche_{metric}_comparison.{ext}"
        fig.savefig(out)
    print(f"Saved niche_{metric}_comparison.svg / .pdf / .png")

plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Weighted average NMI/ARI across datasets (weighted by number of cells)
# with significance brackets comparing each method against TERRA
# ---------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, ttest_rel


CELL_COUNTS = {
    "Sub_molecular_tissue_region": 42136,  # STARmap
    "ccf_region_name": 44686,              # MERFISH
}

total_cells = sum(CELL_COUNTS.values())
WEIGHTS = {rk: n / total_cells for rk, n in CELL_COUNTS.items()}


def p_to_stars(p: float) -> str:
    """Convert p-value to significance stars."""
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


def add_sig_bracket(
    ax,
    x1: float,
    x2: float,
    y: float,
    h: float,
    text: str,
    fontsize: float = 7,
):
    """Draw a significance bracket between two bars."""
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        lw=0.7,
        c="0.2",
        clip_on=False,
        zorder=20,
    )

    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="0.2",
        clip_on=False,
        zorder=21,
    )


def get_weighted_scores_by_resolution(
    df: pd.DataFrame,
    method: str,
    metric: str,
) -> pd.Series:
    """
    Return one weighted score per resolution for one method.

    This keeps the resolution as the index so paired tests are correctly
    aligned between TERRA and each comparison method.
    """
    sub = df[df["method"] == method]

    rows = []

    for res in sorted(sub["resolution"].unique()):
        res_sub = sub[sub["resolution"] == res]

        val = 0.0
        has_any_dataset = False

        for rk, w in WEIGHTS.items():
            rk_vals = res_sub.loc[res_sub["region_key"] == rk, metric].values

            if len(rk_vals) > 0:
                val += rk_vals[0] * w
                has_any_dataset = True

        if has_any_dataset:
            rows.append((res, val))

    if len(rows) == 0:
        return pd.Series(dtype=float, name=method)

    scores = pd.Series(
        data=[v for _, v in rows],
        index=[r for r, _ in rows],
        name=method,
        dtype=float,
    )

    scores.index.name = "resolution"

    return scores


def plot_weighted_metric(
    df: pd.DataFrame,
    metric: str,
    method_order: list[str],
    figsize: tuple = (3.0, 3.8),
    reference_method: str = "stjepa_v2",
    test: str = "wilcoxon",
    draw_ns: bool = True,
):
    """
    Vertical bar plot of weighted average metric across datasets,
    with individual per-resolution data points overlaid.

    Adds brackets over the bars comparing each method against TERRA.
    """
    n_methods = len(method_order)
    bar_width = 0.6
    colors = PALETTE[:n_methods]

    fig, ax = plt.subplots(figsize=figsize)

    # -----------------------------------------------------------------------
    # Compute weighted values
    # -----------------------------------------------------------------------
    means = []
    sds = []
    all_vals = []
    weighted_scores = {}

    for m in method_order:
        scores = get_weighted_scores_by_resolution(df, m, metric)
        weighted_scores[m] = scores

        vals = scores.values

        means.append(np.mean(vals) if len(vals) > 0 else 0)
        sds.append(np.std(vals) if len(vals) > 0 else 0)
        all_vals.append(vals)

    x = np.arange(n_methods)

    # -----------------------------------------------------------------------
    # Bars
    # -----------------------------------------------------------------------
    ax.bar(
        x,
        means,
        width=bar_width,
        color=colors,
        edgecolor="white",
        linewidth=0.6,
        alpha=0.75,
        zorder=2,
    )

    # -----------------------------------------------------------------------
    # Error bars: ± 1 SD
    # -----------------------------------------------------------------------
    ax.errorbar(
        x,
        means,
        yerr=sds,
        fmt="none",
        ecolor="0.25",
        elinewidth=0.8,
        capsize=2.5,
        capthick=0.8,
        zorder=3,
    )

    # -----------------------------------------------------------------------
    # Individual data points
    # -----------------------------------------------------------------------
    rng = np.random.default_rng(42)

    for i, vals in enumerate(all_vals):
        n_pts = len(vals)

        if n_pts == 0:
            continue

        jitter = rng.uniform(-0.15, 0.15, size=n_pts)

        ax.scatter(
            np.full(n_pts, i) + jitter,
            vals,
            s=12,
            color="0.15",
            edgecolor="white",
            linewidth=0.3,
            zorder=4,
            alpha=0.65,
        )

    # -----------------------------------------------------------------------
    # Significance brackets versus TERRA
    # -----------------------------------------------------------------------
    if reference_method not in method_order:
        print(
            f"Warning: reference_method={reference_method!r} was not found in method_order.\n"
            f"Available methods are: {method_order}\n"
            "No significance brackets were added."
        )

    else:
        ref_idx = method_order.index(reference_method)
        ref_scores = weighted_scores[reference_method]

        non_empty_vals = [vals for vals in all_vals if len(vals) > 0]

        if len(non_empty_vals) > 0:
            data_max = max(np.max(vals) for vals in non_empty_vals)
            data_min = min(np.min(vals) for vals in non_empty_vals)
            data_range = max(data_max - data_min, 1e-6)

            bracket_height = 0.035 * data_range
            bracket_gap = 0.11 * data_range
            y0 = data_max + 0.08 * data_range

            bracket_count = 0

            # Draw brackets from nearest to farthest to reduce overlap
            comparison_indices = [
                i for i in range(n_methods)
                if i != ref_idx
            ]

            comparison_indices = sorted(
                comparison_indices,
                key=lambda i: abs(i - ref_idx),
            )

            for i in comparison_indices:
                m = method_order[i]
                comp_scores = weighted_scores[m]

                paired = pd.concat(
                    [
                        ref_scores.rename(reference_method),
                        comp_scores.rename("comparison"),
                    ],
                    axis=1,
                    join="inner",
                ).dropna()

                if paired.shape[0] < 2:
                    print(
                        f"Skipping {m} vs {reference_method}: "
                        f"only {paired.shape[0]} paired points."
                    )
                    continue

                a = paired[reference_method].to_numpy(dtype=float)
                b = paired["comparison"].to_numpy(dtype=float)

                if test == "wilcoxon":
                    try:
                        _, p = wilcoxon(a, b)
                    except ValueError:
                        p = 1.0

                elif test == "ttest":
                    _, p = ttest_rel(a, b, nan_policy="omit")

                else:
                    raise ValueError("test must be either 'wilcoxon' or 'ttest'")

                label = p_to_stars(p)

                if label == "ns" and not draw_ns:
                    continue

                y = y0 + bracket_count * bracket_gap

                add_sig_bracket(
                    ax=ax,
                    x1=ref_idx,
                    x2=i,
                    y=y,
                    h=bracket_height,
                    text=label,
                    fontsize=7,
                )

                bracket_count += 1

            if bracket_count > 0:
                ax.set_ylim(
                    data_min - 0.05 * data_range,
                    y0 + bracket_count * bracket_gap + 0.15 * data_range,
                )

    # -----------------------------------------------------------------------
    # Dotted reference line at STJEPA v2 level
    # -----------------------------------------------------------------------
    if "stjepa_v2" in method_order:
        stjepa_idx = method_order.index("stjepa_v2")

        ax.axhline(
            y=means[stjepa_idx],
            color="0.3",
            linestyle="--",
            linewidth=0.8,
            zorder=1,
            alpha=0.7,
        )

    # -----------------------------------------------------------------------
    # Formatting
    # -----------------------------------------------------------------------
    ax.set_title(f"Weighted {metric}", fontweight="medium")

    ax.set_xticks(x)
    ax.set_xticklabels(
        [METHOD_LABELS.get(m, m) for m in method_order],
        rotation=45,
        ha="right",
    )

    ax.tick_params(axis="x", pad=1)
    ax.set_ylabel(f"{metric} (\u2191)")
    ax.set_xlim(-0.5, n_methods - 0.5)

    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    ax.grid(axis="y", linewidth=0.3, alpha=0.5)

    # Leave room for brackets and rotated labels
    fig.subplots_adjust(
        top=0.88,
        bottom=0.22,
        left=0.20,
        right=0.98,
    )

    return fig


# ---------------------------------------------------------------------------
# Generate and save weighted figures
# ---------------------------------------------------------------------------

method_order = [m for m, _ in method_emb_pairs]

print("Methods in method_order:")
print(method_order)

# Pick the correct internal TERRA method key.
# In your current code, TERRA appears to correspond to "stjepa_v2".
if "stjepa_v2" in method_order:
    reference_method = "stjepa_v2"
elif "terra" in method_order:
    reference_method = "terra"
elif "TERRA" in method_order:
    reference_method = "TERRA"
else:
    raise ValueError(
        "Could not find the TERRA reference method in method_order. "
        f"Available methods are: {method_order}"
    )

print(f"Using {reference_method!r} as the TERRA reference method.")

# Compute and save weighted metrics to CSV
weighted_rows = []

for metric in ["NMI", "ARI"]:
    for m in method_order:
        scores = get_weighted_scores_by_resolution(df_all, m, metric)

        for res, val in scores.items():
            weighted_rows.append(
                {
                    "method": m,
                    "resolution": res,
                    "metric": metric,
                    "weighted_score": val,
                }
            )

df_weighted = pd.DataFrame(weighted_rows)

weighted_results_dir = Path(
    "/nfs/team361/sb75/stjepa-reproducibility/artifacts/ablation/results"
)
weighted_results_dir.mkdir(parents=True, exist_ok=True)

weighted_csv = weighted_results_dir / "weighted_niche_identification_all_methods.csv"
df_weighted.to_csv(weighted_csv, index=False)

print(f"Saved weighted results to {weighted_csv}\n")

for metric in ["NMI", "ARI"]:
    fig = plot_weighted_metric(
        df_all,
        metric=metric,
        method_order=method_order,
        reference_method=reference_method,
        test="wilcoxon",
        draw_ns=True,  # set to False if you only want significant brackets
    )

    for ext in ["svg", "pdf", "png"]:
        out = figures_dir / f"niche_weighted_{metric}_comparison.{ext}"
        fig.savefig(out, bbox_inches="tight")

    print(f"Saved niche_weighted_{metric}_comparison.svg / .pdf / .png")

plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Batch integration benchmarking – vertical bar plot (matching niche ID style)
# ---------------------------------------------------------------------------

BATCH_METRIC_DIRECTION = {
    "iLISI": "\u2191",
    "ASW":   "\u2193",
    "MMD":   "\u2193",
}

# Load batch integration CSVs
batch_frames = []
for method, emb_key in method_emb_pairs:
    fpath = (
        base_path / method
        / f"mmb0-1b_smb1-1b_1p_{emb_key}_batch_integration_metrics.csv"
    )
    if not fpath.exists():
        print(f"[WARN] Batch file not found, skipping: {fpath}")
        continue
    df_batch = pd.read_csv(fpath)
    df_batch["method"] = method
    df_batch["emb_key"] = emb_key
    batch_frames.append(df_batch)

if not batch_frames:
    raise FileNotFoundError("No batch integration CSV files were found.")

df_batch_all = pd.concat(batch_frames, ignore_index=True)

df_batch_all.to_csv(results_dir / "batch_integration_all_methods.csv", index=False)
print(f"Saved batch integration dataframe to {results_dir / 'batch_integration_all_methods.csv'}\n")

method_order = [m for m, _ in method_emb_pairs]

for bm in df_batch_all["metric"].unique():
    is_higher_better = BATCH_METRIC_DIRECTION.get(bm, "") == "\u2191"

    n_methods = len(method_order)
    bar_width = 0.6
    colors = PALETTE[:n_methods]

    fig, ax = plt.subplots(
        figsize=(3.0, 2.4),
        constrained_layout=True,
    )

    # Get one score per method
    means = []
    for m in method_order:
        vals = df_batch_all.loc[
            (df_batch_all["metric"] == bm) & (df_batch_all["method"] == m), "score"
        ].astype(float).values
        means.append(float(np.mean(vals)) if len(vals) > 0 else 0)

    x = np.arange(n_methods)

    # --- bars ---
    ax.bar(
        x, means, width=bar_width,
        color=colors, edgecolor="white", linewidth=0.6,
        alpha=0.75, zorder=2,
    )

    # --- dotted reference line at STJEPA v2 level ---
    if "stjepa_v2" in method_order:
        stjepa_idx = method_order.index("stjepa_v2")
        ax.axhline(y=means[stjepa_idx], color="0.3", linestyle="--",
                   linewidth=0.8, zorder=1, alpha=0.7)

    # --- formatting ---
    direction = BATCH_METRIC_DIRECTION.get(bm, "")
    direction = BATCH_METRIC_DIRECTION.get(bm, "")
    ax.set_title(f"Batch Integration – {bm}", fontweight="medium")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [METHOD_LABELS.get(m, m) for m in method_order],
        rotation=45, ha="right",
    )

    # Add symbols above specific bars
    asterisk_offset = {"iLISI": 0.005, "ASW": 0.004, "MMD": 0.002}.get(bm, 0.01)
    dagger_offset = {"iLISI": 0.03, "ASW": 0.017, "MMD": 0.008}.get(bm, 0.01)
    for i, m in enumerate(method_order):
        if m == "graphst":
            ax.annotate("*", xy=(i, means[i] + asterisk_offset),
                        ha="center", va="center", fontsize=12, fontweight="bold", zorder=5)
        elif m == "banksy":
            ax.annotate("\u2020", xy=(i, means[i] + dagger_offset),
                        ha="center", va="center", fontsize=10, fontweight="bold", zorder=5)
    ax.set_ylabel(f"{bm} ({direction})" if direction else bm)
    ax.set_xlim(-0.5, n_methods - 0.5)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    ax.grid(axis="y", linewidth=0.3, alpha=0.5)

    for ext in ["svg", "pdf", "png"]:
        out = figures_dir / f"batch_integration_{bm}.{ext}"
        fig.savefig(out)
    print(f"Saved batch_integration_{bm}.svg / .pdf / .png")

plt.show()